<a href="https://colab.research.google.com/github/goldurroman/A62-Ideation/blob/main/background_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import zipfile
from google.colab import drive

print(">>> Montage de Google Drive...")
drive.mount('/content/drive')
print("[OK] Drive monté.")

SEARCH_ROOT = "/content/drive/MyDrive"
KEYWORDS = ["550", "50", "50"]   # train, val, test

print(">>> Recherche des dossiers YOLO contenant 550-50-50...\n")

matches = []

for root, dirs, files in os.walk(SEARCH_ROOT):
    for d in dirs:
        name = d.lower()
        if all(k in name for k in KEYWORDS):
            matches.append(os.path.join(root, d))

if matches:
    print(">>> Dossiers trouvés :\n")
    for m in matches:
        print(" -", m)
else:
    print(">>> Aucun dossier correspondant trouvé.")


>>> Montage de Google Drive...
Mounted at /content/drive
[OK] Drive monté.
>>> Recherche des dossiers YOLO contenant 550-50-50...

>>> Dossiers trouvés :

 - /content/drive/MyDrive/runs/yolov8m-seg-550-50-50-2026-05-13-02-57


In [ ]:
import os

paths = [
    "/content/isic2018_raw",
    "/content/isic_yolo",
    "/content/isic_yolo/images/train",
    "/content/isic_yolo/labels/train",
    "/content/drive/MyDrive/runs",
    "/content/drive/MyDrive/train.log",
]

print(">>> Vérification des dossiers essentiels\n")

for p in paths:
    if os.path.exists(p):
        if os.path.isdir(p):
            print(f"[OK] Dossier présent : {p}  → {len(os.listdir(p))} éléments")
        else:
            print(f"[OK] Fichier présent : {p}  → taille {os.path.getsize(p)} octets")
    else:
        print(f"[MANQUANT] ❌ {p}")

print("\n>>> Recherche du checkpoint last.pt...\n")

last_ckpt = None
for root, dirs, files in os.walk("/content/drive/MyDrive/runs"):
    for f in files:
        if f == "last.pt":
            last_ckpt = os.path.join(root, f)

if last_ckpt:
    print(f"[OK] Checkpoint trouvé : {last_ckpt}")
else:
    print("[MANQUANT] ❌ Aucun checkpoint trouvé")


>>> Vérification des dossiers essentiels

[MANQUANT] ❌ /content/isic2018_raw
[MANQUANT] ❌ /content/isic_yolo
[MANQUANT] ❌ /content/isic_yolo/images/train
[MANQUANT] ❌ /content/isic_yolo/labels/train
[OK] Dossier présent : /content/drive/MyDrive/runs  → 1 éléments
[OK] Fichier présent : /content/drive/MyDrive/train.log  → taille 204525 octets

>>> Recherche du checkpoint last.pt...

[OK] Checkpoint trouvé : /content/drive/MyDrive/runs/yolov8m-seg-550-50-50-2026-05-13-02-57/weights/last.pt


In [ ]:
import os
import zipfile
from google.colab import drive

print(">>> Montage de Google Drive...")
drive.mount('/content/drive')
print("[OK] Drive monté.")

# ============================================================
# 1. CHEMINS DES DONNÉES
# ============================================================

RAW_DIR = "/content/isic2018_raw"
INPUT_DIR = f"{RAW_DIR}/ISIC2018_Task1-2_Training_Input"
GT_DIR = f"{RAW_DIR}/ISIC2018_Task1_Training_GroundTruth"

ZIP_INPUT = "/content/drive/MyDrive/ISIC2018_Task1-2_Training_Input.zip"
ZIP_GT    = "/content/drive/MyDrive/ISIC2018_Task1_Training_GroundTruth.zip"

os.makedirs(RAW_DIR, exist_ok=True)

print("\n>>> Vérification des données ISIC...")

# ============================================================
# 2. EXTRACTION AUTOMATIQUE SI NÉCESSAIRE
# ============================================================

def extract_if_missing(zip_path, dest_dir):
    if not os.path.exists(dest_dir) or len(os.listdir(dest_dir)) == 0:
        print(f"[INFO] Extraction de {zip_path} vers {dest_dir} ...")
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(RAW_DIR)
        print(f"[OK] Extraction terminée : {dest_dir}")
    else:
        print(f"[OK] Déjà présent : {dest_dir}")

# Vérifie et extrait si nécessaire
extract_if_missing(ZIP_INPUT, INPUT_DIR)
extract_if_missing(ZIP_GT, GT_DIR)

# ============================================================
# 3. VÉRIFICATION DES FICHIERS
# ============================================================

def count_files(path, ext):
    return len([f for f in os.listdir(path) if f.lower().endswith(ext)])

if not os.path.exists(INPUT_DIR):
    raise RuntimeError(f"[ERREUR] Dossier introuvable : {INPUT_DIR}")

if not os.path.exists(GT_DIR):
    raise RuntimeError(f"[ERREUR] Dossier introuvable : {GT_DIR}")

n_imgs = count_files(INPUT_DIR, ".jpg")
n_masks = count_files(GT_DIR, ".png")

print(f"[OK] Images trouvées : {n_imgs}")
print(f"[OK] Masques trouvés : {n_masks}")

if n_imgs == 0:
    raise RuntimeError("[ERREUR] Aucune image .jpg trouvée dans INPUT_DIR.")

if n_masks == 0:
    raise RuntimeError("[ERREUR] Aucun masque .png trouvé dans GT_DIR.")

print("\n>>> Les données ISIC sont prêtes.")

# ============================================================
# 4. VÉRIFICATION DES CHECKPOINTS POUR REPRISE
# ============================================================

RUNS_DIR = "/content/drive/MyDrive/runs"

os.makedirs(RUNS_DIR, exist_ok=True)

print("\n>>> Vérification des checkpoints YOLO pour reprise...")

last_ckpt = None
for root, dirs, files in os.walk(RUNS_DIR):
    for f in files:
        if f == "last.pt":
            last_ckpt = os.path.join(root, f)

if last_ckpt:
    print(f"[OK] Checkpoint trouvé : {last_ckpt}")
    print("    → L'entraînement pourra reprendre automatiquement avec resume=True.")
else:
    print("[INFO] Aucun checkpoint trouvé.")
    print("    → L'entraînement commencera depuis zéro.")

print("\n>>> Préparation terminée. Vous pouvez maintenant exécuter la cellule %%writefile train.py.")


>>> Montage de Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[OK] Drive monté.

>>> Vérification des données ISIC...
[INFO] Extraction de /content/drive/MyDrive/ISIC2018_Task1-2_Training_Input.zip vers /content/isic2018_raw/ISIC2018_Task1-2_Training_Input ...
[OK] Extraction terminée : /content/isic2018_raw/ISIC2018_Task1-2_Training_Input
[INFO] Extraction de /content/drive/MyDrive/ISIC2018_Task1_Training_GroundTruth.zip vers /content/isic2018_raw/ISIC2018_Task1_Training_GroundTruth ...
[OK] Extraction terminée : /content/isic2018_raw/ISIC2018_Task1_Training_GroundTruth
[OK] Images trouvées : 2594
[OK] Masques trouvés : 2594

>>> Les données ISIC sont prêtes.

>>> Vérification des checkpoints YOLO pour reprise...
[OK] Checkpoint trouvé : /content/drive/MyDrive/runs/yolov8m-seg-550-50-50-2026-05-13-02-57/weights/last.pt
    → L'entraînement pourra reprendre automatiquement avec resume=True

In [ ]:
!ps -ef | grep train.py


root       45325   36575  0 22:06 ?        00:00:00 /bin/bash -c ps -ef | grep train.py
root       45327   45325  0 22:06 ?        00:00:00 grep train.py


In [ ]:
!kill -9 109563


In [ ]:
!ps -ef | grep train.py

root      110552   38087  0 02:32 ?        00:00:00 /bin/bash -c ps -ef | grep train.py
root      110554  110552  0 02:32 ?        00:00:00 grep train.py


In [ ]:
print(">>> Nettoyage du dossier de travail /content...")

!rm -rf /content/isic_yolo
!rm -rf /content/runs
!rm -rf /content/train.log

print("[OK] Nettoyage terminé.")
print(">>> Nettoyage des anciens résultats YOLO...")

# Supprimer les anciens runs YOLO
!rm -rf /content/drive/MyDrive/runs

# Supprimer l'ancien log
!rm -f /content/drive/MyDrive/train.log

print("[OK] Nettoyage terminé.")


>>> Nettoyage du dossier de travail /content...
[OK] Nettoyage terminé.
>>> Nettoyage des anciens résultats YOLO...
[OK] Nettoyage terminé.


In [ ]:
%%writefile train.py
import os
import sys
import glob
import cv2
import numpy as np
from tqdm import tqdm

# ============================================================
# LOGGING (Tee → train.log)
# ============================================================

LOGFILE = "/content/drive/MyDrive/train.log"

class Tee:
    def __init__(self, logfile):
        self.logfile = open(logfile, "a")
        self.stdout = sys.stdout

    def write(self, msg):
        self.stdout.write(msg)
        self.logfile.write(msg)
        self.logfile.flush()
        os.fsync(self.logfile.fileno())

    def flush(self):
        self.stdout.flush()
        self.logfile.flush()
        os.fsync(self.logfile.fileno())

sys.stdout = Tee(LOGFILE)
sys.stderr = Tee(LOGFILE)

def log(msg):
    print(msg)

log("============================================================")
log(">>> train.py : démarrage du script")
log("============================================================")

# ============================================================
# INSTALLATION DES DEPENDANCES
# ============================================================

log(">>> Installation des dépendances (ultralytics, opencv...)")
os.system("pip install ultralytics opencv-python scikit-image matplotlib numpy tqdm --quiet")
log("[OK] Dépendances installées.")

from ultralytics import YOLO

# ============================================================
# CHEMINS
# ============================================================

RAW_DIR = "/content/isic2018_raw"
TRAIN_INPUT_DIR = f"{RAW_DIR}/ISIC2018_Task1-2_Training_Input"
GT_DIR = f"{RAW_DIR}/ISIC2018_Task1_Training_GroundTruth"

BASE_DIR = "/content/isic_yolo"
IMAGES_TRAIN = f"{BASE_DIR}/images/train"
IMAGES_VAL   = f"{BASE_DIR}/images/val"
IMAGES_TEST  = f"{BASE_DIR}/images/test"

LABELS_TRAIN = f"{BASE_DIR}/labels/train"
LABELS_VAL   = f"{BASE_DIR}/labels/val"
LABELS_TEST  = f"{BASE_DIR}/labels/test"

RUNS_DIR = "/content/drive/MyDrive/runs"
os.makedirs(RUNS_DIR, exist_ok=True)

# ============================================================
# VERIFICATION DES DONNEES
# ============================================================

log(">>> Vérification des données ISIC...")

train_imgs_list = sorted(glob.glob(TRAIN_INPUT_DIR + "/*.jpg"))
gt_list = sorted(glob.glob(GT_DIR + "/*.png"))

if len(train_imgs_list) == 0 or len(gt_list) == 0:
    log("[ERREUR] Données manquantes. Exécute la cellule préparatoire.")
    sys.exit(1)

log(f"[OK] Images trouvées : {len(train_imgs_list)}")
log(f"[OK] Masques trouvés : {len(gt_list)}")

# ============================================================
# SPLIT (modifiable librement)
# ============================================================

N_TRAIN = 550
N_VAL   = 50
N_TEST  = 50

np.random.seed(42)
np.random.shuffle(train_imgs_list)

train_imgs = train_imgs_list[:N_TRAIN]
val_imgs   = train_imgs_list[N_TRAIN:N_TRAIN+N_VAL]
test_imgs  = train_imgs_list[N_TRAIN+N_VAL:N_TRAIN+N_VAL+N_TEST]

log(f"[OK] Split : train={len(train_imgs)}, val={len(val_imgs)}, test={len(test_imgs)}")

# ============================================================
# CREATION DES DOSSIERS YOLO
# ============================================================

for d in [IMAGES_TRAIN, IMAGES_VAL, IMAGES_TEST, LABELS_TRAIN, LABELS_VAL, LABELS_TEST]:
    os.makedirs(d, exist_ok=True)

# ============================================================
# PRETRAITEMENT
# ============================================================

def preprocess_image(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (9, 9))
    blackhat = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, kernel)
    _, thresh = cv2.threshold(blackhat, 10, 255, cv2.THRESH_BINARY)
    img = cv2.inpaint(img, thresh, 3, cv2.INPAINT_TELEA)

    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    l2 = clahe.apply(l)
    img = cv2.cvtColor(cv2.merge((l2, a, b)), cv2.COLOR_LAB2BGR)

    img = cv2.resize(img, (256, 256), interpolation=cv2.INTER_AREA)
    return img

def copy_and_preprocess(img_list, dest):
    for p in tqdm(img_list):
        img = cv2.imread(p)
        if img is None:
            continue
        img = preprocess_image(img)
        out = os.path.join(dest, os.path.basename(p))
        cv2.imwrite(out, img)

log(">>> Prétraitement des images...")
copy_and_preprocess(train_imgs, IMAGES_TRAIN)
copy_and_preprocess(val_imgs, IMAGES_VAL)
copy_and_preprocess(test_imgs, IMAGES_TEST)
log("[OK] Prétraitement terminé.")

# ============================================================
# GENERATION DES LABELS
# ============================================================

def img_to_mask(img_path):
    name = os.path.basename(img_path).replace(".jpg", "_segmentation.png")
    return f"{GT_DIR}/{name}"

def mask_to_yolo(mask_path, w, h):
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return []

    _, bin_mask = cv2.threshold(mask, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    contours, _ = cv2.findContours(bin_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)

    polys = []
    for cnt in contours:
        approx = cv2.approxPolyDP(cnt, 0.01 * cv2.arcLength(cnt, True), True)
        if len(approx) < 3:
            continue

        poly = []
        for p in approx:
            x = p[0][0] / w
            y = p[0][1] / h
            poly.extend([x, y])

        polys.append("0 " + " ".join([f"{v:.6f}" for v in poly]))

    return polys

def create_labels(img_list, label_dir):
    for img_path in tqdm(img_list):
        img = cv2.imread(img_path)
        if img is None:
            continue
        h, w = img.shape[:2]
        mask_path = img_to_mask(img_path)
        polys = mask_to_yolo(mask_path, w, h)
        label_path = os.path.join(label_dir, os.path.basename(img_path).replace(".jpg", ".txt"))
        with open(label_path, "w") as f:
            f.write("\n".join(polys))

log(">>> Génération des labels YOLO...")
create_labels(train_imgs, LABELS_TRAIN)
create_labels(val_imgs, LABELS_VAL)
create_labels(test_imgs, LABELS_TEST)
log("[OK] Labels générés.")

# ============================================================
# YAML
# ============================================================

yaml = """
path: /content/isic_yolo
train: images/train
val: images/val
test: images/test

names:
  0: lesion
"""

with open("isic2018-seg.yaml", "w") as f:
    f.write(yaml)

# ============================================================
# NOM DU RUN (stable)
# ============================================================

model_name = "yolov8m-seg"  # change ici pour yolov8l-seg, yolov8n-seg, etc.
train_n = len(train_imgs)
val_n   = len(val_imgs)
test_n  = len(test_imgs)

run_name = f"{model_name}-runs-{train_n}-{val_n}-{test_n}"
run_dir = os.path.join(RUNS_DIR, run_name)
os.makedirs(run_dir, exist_ok=True)

log(f">>> Dossier stable : {run_dir}")

# ============================================================
# REPRISE STABLE (FORCÉE)
# ============================================================

stable_last = os.path.join(run_dir, "weights", "last.pt")

if os.path.exists(stable_last):
    log(f"[OK] Reprise stable depuis : {stable_last}")
    model = YOLO(stable_last)
    RESUME = True
else:
    log("[INFO] Aucun checkpoint stable trouvé. Entraînement depuis zéro.")
    model = YOLO(model_name + ".pt")
    RESUME = False

# ============================================================
# ENTRAINEMENT YOLO
# ============================================================

results = model.train(
    data="isic2018-seg.yaml",
    epochs=200,
    imgsz=256,
    batch=1,
    workers=0,
    augment=True,
    deterministic=True,
    patience=30,
    lr0=0.0005,
    optimizer="AdamW",
    cos_lr=True,
    lrf=0.05,
    weight_decay=0.001,
    mask_ratio=2,
    dropout=0.05,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=15,
    translate=0.1,
    scale=0.15,
    shear=0.1,
    flipud=0.3,
    fliplr=0.5,
    save_period=5,
    project=RUNS_DIR,
    name=run_name,
    resume=RESUME
)

log("[OK] Entraînement terminé.")
log(f"[OK] Résultats dans : {run_dir}")
log("============================================================")
log(">>> FIN DU SCRIPT train.py")
log("============================================================")


Writing train.py


In [ ]:
!sed -n '1,40p' train.py


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


print(">>> Lancement de train.py en arrière-plan...")
!nohup python train.py >/dev/null 2>&1 &


import time
time.sleep(1)

print("\n>>> Vérification des fichiers dans /content :")
!ls -l /content

print("\n>>> Processus Python actifs :")
!ps -ef | grep python



In [ ]:
print(">>> Vérification de l'existence du fichier train.log :")
!ls -l /content/drive/MyDrive/train.log

print("\n>>> Dernières lignes du log :")
#!tail -n 500 /content/drive/MyDrive/train.log
!cat /content/drive/MyDrive/train.log | tail -n 80


In [ ]:
# ============================================================
# MONITORING MANUEL DU LOG D'ENTRAÎNEMENT
# ============================================================

LOGFILE = "/content/drive/MyDrive/train.log"

print(">>> Vérification de l'existence du fichier train.log :")
!ls -l /content/drive/MyDrive/train.log 2>/dev/null || echo "[MANQUANT] Aucun fichier train.log trouvé."

print("\n>>> Dernières lignes du log :")
!cat /content/drive/MyDrive/train.log 2>/dev/null | tail -n 80 || echo "[INFO] Le log n'existe pas encore ou est vide."


In [ ]:
# ============================================================
# AUTO-MONITORING DU LOG D'ENTRAÎNEMENT (EN TEMPS RÉEL)
# ============================================================

import time
import os

LOGFILE = "/content/drive/MyDrive/train.log"
INTERVAL = 10  # secondes

print(">>> Auto-monitor du log (Notebook de train)")
print(">>> Rafraîchissement toutes les", INTERVAL, "secondes")
print(">>> Ctrl+C pour arrêter")
print("------------------------------------------------------------")

try:
    while True:
        print("\n============================================================")
        print(">>> Mise à jour :", time.strftime("%H:%M:%S"))
        print("============================================================")

        if os.path.exists(LOGFILE):
            !cat /content/drive/MyDrive/train.log | tail -n 80
        else:
            print("[INFO] Le fichier train.log n'existe pas encore.")

        time.sleep(INTERVAL)

except KeyboardInterrupt:
    print("\n>>> Auto-monitor arrêté.")
